# Understanding CUDA Profiling

Importing Torch and checking the version of it

In [1]:
!nvidia-smi

Wed Jun 17 17:20:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
import time
print("torch version: ",torch.__version__)


torch version:  2.8.0+cu128


Running a simple `torch.square` operation

In [3]:
a=torch.tensor([2,1])
a

tensor([2, 1])

In [4]:
b=torch.square(a)
b

tensor([4, 1])

In [5]:
x=torch.tensor(2.5)

In [6]:
start = time.time()
y=torch.square(x)
end = time.time()
print("Time taken for torch.square call: ", end-start)

Time taken for torch.square call:  0.0002505779266357422


This time is fundamentally wrong since the CPU time difference counts the kernel launch overhead when we actually want the GPU time difference.

One thing you have clearly observe is that the CPU time never actually takes into account the actual GPU time -- meaning that the GPU/CUDA Kernel is being run in `async`. So we have to synchronize the CPU time in a way such that the GPU time is taken into consideration...

So we can use multiple ways to assess the GPU time better than using the CPU's time library

## CUDA Events

In [7]:
print(torch.cuda)
print(torch.cuda.Event)

<module 'torch.cuda' from '/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/cuda/__init__.py'>
<class 'torch.cuda.streams.Event'>


In [8]:
start_cuda = torch.cuda.Event(enable_timing=True)
end_cuda = torch.cuda.Event(enable_timing=True)

In [9]:
print(start_cuda)
print(end_cuda)

<torch.cuda.Event uninitialized>
<torch.cuda.Event uninitialized>


Look at the above these events are `uninitialized`!

In [10]:
start_cuda.record()
y=torch.square(x)
end_cuda.record()

In [11]:
print(torch.cuda.synchronize)

<function synchronize at 0x76d90a14ba60>


In [12]:
torch.cuda.synchronize() #forces the CPU to wait until all previously queued CUDA work on the GPU has finished.

In [13]:
print(start_cuda)
print(end_cuda)

<torch.cuda.Event 0x94c14e0>
<torch.cuda.Event 0x94bd600>


In [14]:
print("Time Elapsed: ", start_cuda.elapsed_time(end_cuda))

Time Elapsed:  0.3553279936313629


## Pytorch Autograd Profiler

In [15]:
help(torch.randn)

Help on built-in function randn in module torch:

randn(...)
    randn(*size, *, generator=None, out=None, dtype=None, layout=torch.strided, device=None, requires_grad=False, pin_memory=False) -> Tensor


    Returns a tensor filled with random numbers from a normal distribution
    with mean `0` and variance `1` (also called the standard normal
    distribution).

    .. math::
        \text{out}_{i} \sim \mathcal{N}(0, 1)

    For complex dtypes, the tensor is i.i.d. sampled from a `complex normal distribution`_ with zero mean and
    unit variance as

    .. math::
        \text{out}_{i} \sim \mathcal{CN}(0, 1)

    This is equivalent to separately sampling the real :math:`(\operatorname{Re})` and imaginary
    :math:`(\operatorname{Im})` part of :math:`\text{out}_i` as

    .. math::
        \operatorname{Re}(\text{out}_{i}) \sim \mathcal{N}(0, \frac{1}{2}),\quad
        \operatorname{Im}(\text{out}_{i}) \sim \mathcal{N}(0, \frac{1}{2})

    The shape of the tensor is defined by th

In [16]:
x = torch.randn((1024,1024),device='cuda')

In [17]:
x

tensor([[ 0.9998, -0.1651, -1.3194,  ...,  0.4540,  0.0810,  0.4620],
        [ 1.0559, -0.2608,  1.4241,  ..., -0.3898, -0.1833,  0.7462],
        [ 1.6242,  0.6363,  1.7869,  ..., -1.8981, -1.2612,  0.0871],
        ...,
        [ 0.1723, -0.5727,  0.8832,  ...,  0.9890, -1.0094, -0.4180],
        [ 0.8276,  0.0740, -0.8599,  ...,  0.5923, -0.6900,  0.7364],
        [-0.9445,  0.3071, -0.9953,  ...,  1.2504, -0.5774, -1.2284]],
       device='cuda:0')

In [18]:
with torch.autograd.profiler.profile(use_device='cuda') as prof:
    y = torch.square(x)

In [19]:
print(prof)

-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
          cudaEventRecord         0.04%      14.795us         0.04%      14.795us      14.795us       0.000us         0.00%       0.000us       0.000us             1  
             aten::square         0.16%      55.187us        99.93%      35.563ms      35.563ms      51.000us         0.14%      35.570ms      35.570ms             1  
          cudaEventRecord         0.01%       3.457us         0.01%       3.457us       3.457us       0.000us         0.00%       0.000us       0.000us        

In [20]:
print(prof.key_averages().table(sort_by="cuda_time_total"))

-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
             aten::square         0.16%      55.187us        99.93%      35.563ms      35.563ms      51.000us         0.14%      35.570ms      35.570ms             1  
                aten::pow         0.40%     143.546us        99.75%      35.498ms      35.498ms      35.500ms        99.80%      35.519ms      35.519ms             1  
        aten::result_type         0.01%       2.927us         0.01%       2.927us       2.927us      13.000us         0.04%      13.000us      13.000us        

## Pytorch Profiler with Chrome Trace

In [21]:
from torch.profiler import profile, ProfilerActivity

x_cpu = torch.randn((1024,1024))

In [22]:
with profile(activities=[ProfilerActivity.CPU,ProfilerActivity.CUDA], record_shapes=True) as chrprof:
    x_gpu = x_cpu.to("cuda")
    y = torch.square(x_gpu)

chrprof.export_chrome_trace("trace.json")

Then you can comfortably understand the activities happening on local chrome trace with ease

## NVIDIA Nsight Compute

> Make sure you have installed `ninja` library

In [23]:
!pip install ninja

In [24]:
from torch.utils.cpp_extension import load_inline

In [25]:
cpp_source = """
#include <string>

std::string hello_world() {
    return "hello world";
}
"""


In [26]:
cpp_source

'\n#include <string>\n\nstd::string hello_world() {\n    return "hello world";\n}\n'

In [27]:
module = load_inline(
    name = "hello_world",
    cpp_sources=cpp_source,
    functions=["hello_world"],
    verbose=True
)


[1/2] c++ -MMD -MF main.o.d -DTORCH_EXTENSION_NAME=hello_world -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1018\" -isystem /home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/include -isystem /home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/include/torch/csrc/api/include -isystem /home/zeus/miniconda3/envs/cloudspace/include/python3.12 -fPIC -std=c++17 -c /teamspace/studios/this_studio/.cache/torch_extensions/py312_cu128/hello_world/main.cpp -o main.o 
[2/2] c++ main.o -shared -L/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o hello_world.so


In [28]:
print(module.hello_world())

hello world


Now implementing inline_load with a kernel function (inline)

In [29]:
cpp_source = r"""
#include <torch/extension.h>

torch::Tensor square_cuda(torch::Tensor input);
"""

cuda_source = r"""
#include <torch/extension.h>

__global__ void square_kernel(const float* input, float* output, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        float x = input[idx];
        output[idx] = x * x;
    }
}

torch::Tensor square_cuda(torch::Tensor input) {
    input = input.contiguous();

    auto output = torch::empty_like(input);

    int n = input.numel();
    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    square_kernel<<<blocks, threads>>>(
        input.data_ptr<float>(),
        output.data_ptr<float>(),
        n
    );

    return output;
}
"""

module = load_inline(
    name="square_cuda_extension_clean_v2",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=["square_cuda"],
    extra_cuda_cflags=["-O2"],
    verbose=True,
)

[1/3] c++ -MMD -MF main.o.d -DTORCH_EXTENSION_NAME=square_cuda_extension_clean_v2 -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1018\" -isystem /home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/include -isystem /home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /home/zeus/miniconda3/envs/cloudspace/include/python3.12 -fPIC -std=c++17 -c /teamspace/studios/this_studio/.cache/torch_extensions/py312_cu128/square_cuda_extension_clean_v2/main.cpp -o main.o 
[2/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output cuda.cuda.o.d -DTORCH_EXTENSION_NAME=square_cuda_extension_clean_v2 -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1018\" -isystem /home/zeus/miniconda3/envs/cloudspace/lib

In [30]:
x = torch.randn(1024, 1024, device="cuda", dtype=torch.float32)
y = module.square_cuda(x)

print(torch.allclose(y, x * x))
print(y[:5, :5])

True
tensor([[1.2749e-01, 2.4162e+00, 2.5131e-01, 1.0281e+00, 1.3881e+00],
        [1.5525e+00, 1.1481e+00, 5.5917e-01, 1.5415e-01, 4.6909e-05],
        [2.1631e+00, 3.2903e-02, 5.5619e-03, 3.3099e+00, 1.9394e-01],
        [8.1468e+00, 3.4960e-01, 1.3480e+00, 1.0787e+00, 7.4123e-01],
        [1.6346e-02, 3.9254e+00, 1.1134e-02, 7.5898e-02, 8.3349e-02]],
       device='cuda:0')


## Triton

In [31]:
!pip install triton

In [32]:
import triton
import triton.language as tl

In [34]:
help(triton.jit)

Help on function jit in module triton.runtime.jit:

jit(fn: 'Optional[T]' = None, *, version=None, repr: 'Optional[Callable]' = None, launch_metadata: 'Optional[Callable]' = None, do_not_specialize: 'Optional[Iterable[int | str]]' = None, do_not_specialize_on_alignment: 'Optional[Iterable[int | str]]' = None, debug: 'Optional[bool]' = None, noinline: 'Optional[bool]' = None) -> 'Union[JITFunction[T], Callable[[T], JITFunction[T]]]'
    Decorator for JIT-compiling a function using the Triton compiler.

    :note: When a jit'd function is called, arguments are
        implicitly converted to pointers if they have a :code:`.data_ptr()` method
        and a `.dtype` attribute.

    :note: This function will be compiled and run on the GPU. It will only have access to:

           * python primitives,
           * builtins within the triton package,
           * arguments to this function,
           * other jit'd functions

    :param fn: the function to be jit-compiled
    :type fn: Callab

`@triton.jit` tells Triton: “this Python function is not normal Python anymore; treat it as a GPU kernel and compile it.” When you first call the function with a launch grid like `square_kernel[grid](...)`, Triton traces/compiles the function into GPU code through its compiler pipeline, specializing it for compile-time constants like `BLOCK_SIZE`, tensor dtypes, and target GPU architecture. So your Python-looking code using `tl.load`, `tl.arange`, `tl.store`, etc. becomes low-level GPU code such as PTX/CUBIN. The first call may be slower because compilation happens, but later calls reuse the cached compiled kernel.


In [51]:
help(triton.cdiv)

Help on function cdiv in module triton:

cdiv(x: int, y: int)



In [53]:
help(tl.load)

Help on function load in module triton.language.core:

load(pointer, mask=None, other=None, boundary_check=(), padding_option='', cache_modifier='', eviction_policy='', volatile=False, _semantic=None)
    Return a tensor of data whose values are loaded from memory at location defined by `pointer`:

        (1) If `pointer` is a single element pointer, a scalar is be loaded.  In
            this case:

            - `mask` and `other` must also be scalars,
            - `other` is implicitly typecast to `pointer.dtype.element_ty`, and
            - `boundary_check` and `padding_option` must be empty.

        (2) If `pointer` is an N-dimensional tensor of pointers, an
            N-dimensional tensor is loaded.  In this case:

            - `mask` and `other` are implicitly broadcast to `pointer.shape`,
            - `other` is implicitly typecast to `pointer.dtype.element_ty`, and
            - `boundary_check` and `padding_option` must be empty.

        (3) If `pointer` is a block po

In [54]:
help(tl.store)

Help on function store in module triton.language.core:

store(pointer, value, mask=None, boundary_check=(), cache_modifier='', eviction_policy='', _semantic=None)
    Store a tensor of data into memory locations defined by `pointer`.

        (1) If `pointer` is a single element pointer, a scalar is stored.  In
            this case:

            - `mask` must also be scalar, and
            - `boundary_check` and `padding_option` must be empty.

        (2) If `pointer` is an N-dimensional tensor of pointers, an
            N-dimensional block is stored.  In this case:

            - `mask` is implicitly broadcast to `pointer.shape`, and
            - `boundary_check` must be empty.

        (3) If `pointer` is a block pointer defined by `make_block_ptr`, a block
            of data is stored.  In this case:

            - `mask` must be None, and
            - `boundary_check` can be specified to control the behavior of out-of-bound access.

    `value` is implicitly broadcast to `po

In [41]:
@triton.jit 
# Triton JIT Function
def square_kernel(x_ptr, y_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0) # gives program id to the instance
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE) 
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask) 
    y=x*x 
    tl.store(y_ptr + offsets, y, mask=mask)

# Python Launcher for Square Kernel
def square_triton(x):
    y = torch.empty_like(x) 
    n_elements = x.numel()
    grid = lambda meta: (
        triton.cdiv(n_elements, meta["BLOCK_SIZE"]),
    )
    square_kernel[grid](
        x, y, n_elements, BLOCK_SIZE=1024
    )
    return y 

In [42]:
x = torch.randn(1024, 1024, device="cuda", dtype=torch.float32)

y_triton = square_triton(x)
y_torch = x * x

print(torch.allclose(y_triton, y_torch))
print(torch.max(torch.abs(y_triton - y_torch)))

True
tensor(0., device='cuda:0')


In [40]:
help(torch.allclose)

Help on built-in function allclose in module torch:

allclose(...)
    allclose(input: Tensor, other: Tensor, rtol: float = 1e-05, atol: float = 1e-08, equal_nan: bool = False) -> bool

    This function checks if :attr:`input` and :attr:`other` satisfy the condition:

    .. math::
        \lvert \text{input}_i - \text{other}_i \rvert \leq \texttt{atol} + \texttt{rtol} \times \lvert \text{other}_i \rvert

    elementwise, for all elements of :attr:`input` and :attr:`other`. The behaviour of this function is analogous to
    `numpy.allclose <https://numpy.org/doc/stable/reference/generated/numpy.allclose.html>`_

    Args:
        input (Tensor): first tensor to compare
        other (Tensor): second tensor to compare
        atol (float, optional): absolute tolerance. Default: 1e-08
        rtol (float, optional): relative tolerance. Default: 1e-05
        equal_nan (bool, optional): if ``True``, then two ``NaN`` s will be considered equal. Default: ``False``

    Example::

        >

In [43]:
def benchmark(fn, x, iters=100):
    # warmup
    for _ in range(10):
        y = fn(x)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()
    for _ in range(iters):
        y = fn(x)
    end.record()

    torch.cuda.synchronize()

    return start.elapsed_time(end) / iters  # milliseconds

In [44]:
x = torch.randn(10_000_000, device="cuda", dtype=torch.float32)

triton_ms = benchmark(square_triton, x)
torch_ms = benchmark(lambda z: z * z, x)

print(f"Triton square: {triton_ms:.4f} ms")
print(f"PyTorch x*x:   {torch_ms:.4f} ms")

Triton square: 0.3472 ms
PyTorch x*x:   0.3468 ms


You can also debug triton kernels with `interpret=True` on the JIT decorator...

Let's try reading and interpreting the PTX (note that Triton directly generates IRs instead of C++ Code!) so we need to be able to read/interpret PTX Instructions

In [49]:
x = torch.randn(1024, device="cuda")
y = square_triton(x)
torch.cuda.synchronize()

In [50]:
print(x)
print(y)

tensor([ 0.5860,  2.3935, -1.7863,  ...,  0.4883, -0.1898,  0.0868],
       device='cuda:0')
tensor([0.3434, 5.7291, 3.1907,  ..., 0.2385, 0.0360, 0.0075], device='cuda:0')


In [55]:
print(x.numel())

1024


In [57]:
x2=torch.randn((1024,1024))
print(x2.numel())

1048576


In [58]:
help(torch.numel)

Help on built-in function numel in module torch:

numel(...)
    numel(input: Tensor) -> int

    Returns the total number of elements in the :attr:`input` tensor.

    Args:
        input (Tensor): the input tensor.

    Example::

        >>> a = torch.randn(1, 2, 3, 4, 5)
        >>> torch.numel(a)
        120
        >>> a = torch.zeros(4,4)
        >>> torch.numel(a)
        16



In [46]:
compiled = square_kernel[(triton.cdiv(x.numel(), 1024),)](
    x, y, x.numel(),
    BLOCK_SIZE=1024
)

After square_triton(x) has already compiled the kernel, this line is just manually launching the same Triton GPU kernel again, without using your Python wrapper.

It means: launch square_kernel with a grid of ceil(num_elements / 1024) Triton programs, pass x as input, y as output, pass the element count, and use the compile-time constant BLOCK_SIZE=1024. Since the kernel was already compiled for this specialization, Triton likely reuses the cached compiled code and only launches it. Also, compiled is a misleading variable name here: this call usually returns None; the real result is written into y on the GPU.

In [47]:
print(compiled.asm["ptx"])

//
// Generated by LLVM NVPTX Back-End
//

.version 8.7
.target sm_89
.address_size 64

	// .globl	square_kernel           // -- Begin function square_kernel
                                        // @square_kernel
.visible .entry square_kernel(
	.param .u64 .ptr .global .align 1 square_kernel_param_0,
	.param .u64 .ptr .global .align 1 square_kernel_param_1,
	.param .u32 square_kernel_param_2,
	.param .u64 .ptr .global .align 1 square_kernel_param_3
)
.reqntid 128
{
	.reg .pred 	%p<5>;
	.reg .b32 	%r<25>;
	.reg .b64 	%rd<8>;
	.loc	1 3 0                           // 1090843465.py:3:0
$L__func_begin0:
	.loc	1 3 0                           // 1090843465.py:3:0

// %bb.0:
	ld.param.b64 	%rd5, [square_kernel_param_0];
	ld.param.b64 	%rd6, [square_kernel_param_1];
$L__tmp0:
	.loc	1 4 24                          // 1090843465.py:4:24
	mov.u32 	%r17, %ctaid.x;
	.loc	1 5 20                          // 1090843465.py:5:20
	shl.b32 	%r18, %r17, 10;
	ld.param.b32 	%r19, [square_kernel_param_2];
	

In [48]:
print(compiled)

## torch.compile()

`torch.compile` takes a normal PyTorch function or model and tries to make it run faster by capturing its operations into a graph, optimizing that graph, and generating more efficient backend code. Instead of executing every PyTorch op one by one through Python, it can fuse operations, reduce overhead, and choose optimized kernels through backends like TorchInductor. The first call may be slower because compilation happens, but later calls can be faster because the optimized version is reused.


In [65]:
# OG defintion
def square_fn(a):
    a=torch.square(a)
    return a

compiled_sqaure = torch.compile(square_fn) # compiled function

In [66]:
x=torch.randn(1024, device="cuda")

In [68]:
with torch.autograd.profiler.profile(use_device='cuda') as prof_compiled:
    y = compiled_sqaure(x)

In [69]:
print(prof_compiled)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        cudaEventRecord         0.03%      17.292us         0.03%      17.292us      17.292us       0.000us         0.00%       0.000us       0.000us             1  
                               TorchDynamo Cache Lookup         0.01%       5.763us         0.01%       5.763us       5.763us      14.000us         0.03%      14.000us      14.000us             1  
         

In [71]:
print(prof_compiled.key_averages().table(sort_by="cuda_time_total"))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                  _compile.compile_inner (dynamo_timed)         2.50%       1.394ms        99.11%      55.380ms      55.380ms       1.366ms         2.45%      55.400ms      55.400ms             1  
                            build_guards (dynamo_timed)        49.47%      27.647ms        49.47%      27.647ms      27.647ms      27.696ms        49.59%      27.696ms      27.696ms             1  
         